# 🌿 Scikit-Learn — Predictive AI Notebook

End-to-end predictive AI workflows using scikit-learn, from raw data to deployed predictions.

**Real-world examples:**
1. Credit default prediction (binary classification)
2. House price regression
3. Customer churn with SHAP explanations
4. Time-series demand forecasting
5. Anomaly detection for IoT sensors
6. Multi-label product tag prediction
7. Model comparison and selection
8. Production pipeline serialisation

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, TimeSeriesSplit)
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               HistGradientBoostingClassifier, IsolationForest)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (roc_auc_score, classification_report,
                              mean_absolute_error, r2_score)
from sklearn.multioutput import MultiOutputClassifier
import joblib

np.random.seed(42)
print('Scikit-learn ready.')

## 1. Credit Default Prediction — Binary Classification

In [ ]:
# ── Generate Synthetic Credit Data ──────────────────────────────────────────
# Simulates a typical lending dataset with mixed feature types
np.random.seed(0)
n = 5000

df_credit = pd.DataFrame({
    'age':             np.random.randint(18, 70, n),
    'income':          np.random.lognormal(10.5, 0.5, n),          # right-skewed
    'loan_amount':     np.random.uniform(1000, 50000, n),
    'credit_history':  np.random.randint(0, 120, n),               # months
    'num_accounts':    np.random.randint(1, 10, n),
    'employment_type': np.random.choice(['employed','self','unemployed'], n,
                                        p=[0.6, 0.25, 0.15]),
    'purpose':         np.random.choice(['home','car','education','personal'], n),
})

# Introduce 3% missing values in income and credit_history
df_credit.loc[np.random.choice(n, int(n*0.03), replace=False), 'income'] = np.nan
df_credit.loc[np.random.choice(n, int(n*0.02), replace=False), 'credit_history'] = np.nan

# Target: default (1=default, 0=repaid)
# Higher probability of default for: higher loan, lower income, unemployed
log_odds = (-2 + 0.01 * df_credit['loan_amount'] / df_credit['income'].fillna(df_credit['income'].median())
            + (df_credit['employment_type'] == 'unemployed').astype(float) * 1.5
            - 0.01 * df_credit['credit_history'].fillna(60))
prob_default = 1 / (1 + np.exp(-log_odds))
df_credit['default'] = (np.random.random(n) < prob_default).astype(int)

print(df_credit.head(3).to_string())
print(f'\nDefault rate: {df_credit["default"].mean()*100:.1f}%')

# ── Build Pipeline ────────────────────────────────────────────────────────────
num_features = ['age','income','loan_amount','credit_history','num_accounts']
cat_features = ['employment_type','purpose']

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),  # fill missing with median
    ('scale',  RobustScaler()),                    # outlier-resistant scaling
])

categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe,    num_features),
    ('cat', categorical_pipe, cat_features),
])

credit_pipeline = Pipeline([
    ('prep',  preprocessor),
    ('model', HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, early_stopping=True,
        validation_fraction=0.1, random_state=42)),
])

X = df_credit.drop(columns='default')
y = df_credit['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

credit_pipeline.fit(X_train, y_train)
y_prob = credit_pipeline.predict_proba(X_test)[:, 1]
print(f'\nCredit Default AUC: {roc_auc_score(y_test, y_prob):.4f}')
print(classification_report(y_test, credit_pipeline.predict(X_test)))

## 2. House Price Regression with Feature Engineering

In [ ]:
# ── House Price Regression ────────────────────────────────────────────────────
np.random.seed(1)
n = 2000

df_houses = pd.DataFrame({
    'sqft':        np.random.uniform(500, 5000, n),
    'bedrooms':    np.random.randint(1, 7, n),
    'bathrooms':   np.random.uniform(1, 4, n).round(1),
    'age_years':   np.random.randint(0, 80, n),
    'lot_sqft':    np.random.uniform(2000, 20000, n),
    'garage':      np.random.randint(0, 4, n),
    'neighborhood':np.random.choice(['A','B','C','D'], n, p=[0.3,0.3,0.25,0.15]),
    'condition':   np.random.choice(['poor','fair','good','excellent'], n),
})

# Price model: base + neighbourhood premium + condition premium + sqft × quality
nb_premium = {'A': 50000, 'B': 25000, 'C': 0, 'D': -30000}
cond_mult  = {'poor': 0.7, 'fair': 0.85, 'good': 1.0, 'excellent': 1.2}
price = (100 * df_houses['sqft']
         + df_houses['neighborhood'].map(nb_premium)
         + df_houses['condition'].map(cond_mult) * 20000
         - 500 * df_houses['age_years']
         + 15000 * df_houses['bathrooms']
         + np.random.normal(0, 15000, n))
df_houses['price'] = price.clip(50000)

X_h = df_houses.drop(columns='price')
y_h = np.log1p(df_houses['price'])   # log transform for better-behaved distribution

num_h = ['sqft','bedrooms','bathrooms','age_years','lot_sqft','garage']
cat_h = ['neighborhood','condition']

house_pipeline = Pipeline([
    ('prep', ColumnTransformer([
        ('num', Pipeline([('sc', StandardScaler())]), num_h),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_h),
    ])),
    ('model', Ridge(alpha=10.0)),
])

# 5-fold cross-validation
cv_r2 = cross_val_score(house_pipeline, X_h, y_h, cv=5, scoring='r2')
print(f'House price CV R² = {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')

house_pipeline.fit(*train_test_split(X_h, y_h, test_size=0.2, random_state=42)[:2])
_, X_h_test, _, y_h_test = train_test_split(X_h, y_h, test_size=0.2, random_state=42)
y_pred_log = house_pipeline.predict(X_h_test)
mae = mean_absolute_error(np.expm1(y_h_test), np.expm1(y_pred_log))
print(f'Test MAE: ${mae:,.0f}')

## 3. IoT Sensor Anomaly Detection

In [ ]:
# ── Anomaly Detection for Industrial IoT Sensors ──────────────────────────────
# Scenario: monitor 4 sensors on a pump. Detect abnormal readings.
np.random.seed(5)
n_normal = 1000

# Normal operating readings
normal_data = np.column_stack([
    np.random.normal(60, 3, n_normal),   # temperature (°C)
    np.random.normal(4.5, 0.2, n_normal),# vibration (mm/s)
    np.random.normal(2.1, 0.15, n_normal),# pressure (bar)
    np.random.normal(1450, 30, n_normal), # RPM
])

# Inject anomalies: 30 instances with abnormal readings
anomalies = np.column_stack([
    np.random.normal(85, 5, 30),   # overheating
    np.random.normal(12, 2, 30),   # high vibration
    np.random.normal(1.2, 0.3, 30),# low pressure
    np.random.normal(1600, 50, 30),# over-speed
])

all_data = np.vstack([normal_data, anomalies])
true_labels = np.array([1]*n_normal + [-1]*30)   # 1=normal, -1=anomaly

# Isolation Forest: trained only on normal data (unsupervised)
iso = IsolationForest(n_estimators=300, contamination=0.03,
                      random_state=42, n_jobs=-1)
iso.fit(normal_data)   # learn the distribution of normal data only

# Score new data: -1 = anomaly, 1 = normal
predictions = iso.predict(all_data)
anomaly_scores = -iso.score_samples(all_data)   # higher = more anomalous

# Evaluation metrics
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(true_labels, predictions, labels=[1, -1])
detected = (predictions[-30:] == -1).sum()
print(f'Anomalies detected: {detected}/30  ({detected/30*100:.0f}% recall)')
print(f'False alarms (normal classified as anomaly): {(predictions[:n_normal]==-1).sum()}')

# Visualise anomaly scores
plt.figure(figsize=(10, 3))
plt.scatter(range(n_normal), anomaly_scores[:n_normal],
            alpha=0.4, s=8, color='#60a5fa', label='Normal')
plt.scatter(range(n_normal, n_normal+30), anomaly_scores[n_normal:],
            alpha=0.9, s=30, color='#ef4444', label='Anomaly')
plt.axhline(np.percentile(anomaly_scores[:n_normal], 97),
            color='#f59e0b', linestyle='--', lw=1.5, label='Threshold (97th %ile)')
plt.xlabel('Sample'); plt.ylabel('Anomaly score')
plt.title('IoT Sensor Anomaly Detection — Isolation Forest')
plt.legend(); plt.tight_layout(); plt.show()

## 4. Model Comparison and Selection

In [ ]:
# ── Model Comparison: Credit Default ─────────────────────────────────────────

# Build multiple candidate models using the same preprocessor
models = {
    'Logistic Regression':      LogisticRegression(C=1.0, max_iter=500),
    'Random Forest':            RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    'Gradient Boosting':        GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42),
    'HistGradientBoosting':     HistGradientBoostingClassifier(max_iter=200, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', clf)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    results[name] = scores
    print(f'{name:30s} AUC = {scores.mean():.4f} ± {scores.std():.4f}')

# Visualise comparison
plt.figure(figsize=(8, 4))
positions = range(len(results))
means = [v.mean() for v in results.values()]
stds  = [v.std()  for v in results.values()]
bars = plt.bar(positions, means, yerr=stds, capsize=5,
               color=['#60a5fa','#34d399','#f59e0b','#a78bfa'], alpha=0.85)
plt.xticks(positions, list(results.keys()), rotation=15, ha='right')
plt.ylabel('ROC-AUC'); plt.ylim(0.7, 1.0)
plt.title('5-Fold CV ROC-AUC Comparison')
for bar, mean in zip(bars, means):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.003,
             f'{mean:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── 5. Save and Load Production Pipeline ─────────────────────────────────────

# Save the best model (entire pipeline including preprocessing)
joblib.dump(credit_pipeline, '/tmp/credit_default_pipeline.pkl', compress=3)
print('Pipeline saved to /tmp/credit_default_pipeline.pkl')

# Load and verify
loaded_pipeline = joblib.load('/tmp/credit_default_pipeline.pkl')
y_prob_loaded   = loaded_pipeline.predict_proba(X_test)[:, 1]
print(f'Loaded pipeline AUC: {roc_auc_score(y_test, y_prob_loaded):.4f}')  # same as before

# Score a new single application
new_app = pd.DataFrame([{
    'age': 35, 'income': 55000, 'loan_amount': 15000,
    'credit_history': 48, 'num_accounts': 3,
    'employment_type': 'employed', 'purpose': 'car'
}])

prob = loaded_pipeline.predict_proba(new_app)[0, 1]
decision = 'APPROVE' if prob < 0.2 else 'MANUAL_REVIEW' if prob < 0.5 else 'DECLINE'
print(f'\nNew application: P(default) = {prob:.4f} → Decision: {decision}')